In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import json
import os
import glob
from datetime import datetime

# =====================================================================
# 🦅 LIVE PRODUCTION TRADE CONFIGURATIONS (Modify your capital here)
# =====================================================================
USER_PORTFOLIO_VALUE = 25000.00  # Total cash dollar balance to invest
# =====================================================================

def find_and_parse_prior_month_file() -> pd.DataFrame:
    """
    Scans the current workspace directory for any reallocation CSV files
    generated during the previous calendar month.
    """
    today = datetime.now()
    
    # Calculate previous month and year values handling January rollbacks safely
    prev_month = 12 if today.month == 1 else today.month - 1
    prev_year = today.year - 1 if today.month == 1 else today.year
    
    # Format pattern to match matching month string sequence (e.g. "05??2026_Reallocations.csv")
    match_pattern = f"{prev_month:02d}??{prev_year}_Reallocations.csv"
    discovered_files = glob.glob(match_pattern)
    
    if not discovered_files:
        print(f"🔍 [Scanner]: No historical reallocations ledger found matching target pattern '{match_pattern}'.")
        return None
        
    # Sort files to isolate the latest logging file if multiple match the pattern
    target_file = sorted(discovered_files)[-1]
    print(f"📖 [Scanner]: Discovered historical portfolio anchor data sheet: {target_file}")
    
    # Read and ensure index tracking is correctly parsed
    df_prior = pd.read_csv(target_file)
    if "Portfolio Position" in df_prior.columns:
        df_prior = df_prior.set_index("Portfolio Position")
    return df_prior


def execute_production_signals_run(portfolio_cash: float):
    target_csv = "Dynamic_etf_optimized_variables.csv"
    
    if not os.path.exists(target_csv):
        raise FileNotFoundError(f"❌ Aborted: Configuration ledger file '{target_csv}' missing. Run optimizer first.")
        
    df_variables = pd.read_csv(target_csv)
    etf_target  = str(df_variables["etf_target"].iloc[0]).upper().strip()
    num_stocks  = int(df_variables["num_stocks"].iloc[0])
    allocations = json.loads(df_variables["allocations"].iloc[0])
    
    total_w_sum = sum(allocations) if sum(allocations) > 0 else 100
    normalized_weights = [w / total_w_sum for w in allocations]
    
    print(f"🎯 Target Model: ETF={etf_target} | Buy Density N={num_stocks} Stocks | Total Capital=${portfolio_cash:,.2f}")
    
    # 2. Extract full-basket composition from index repositories
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print(f"📡 Resolving underlying constituent matrix for {etf_target}...")
    try:
        url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies" if "SPY" in etf_target else "https://en.wikipedia.org/wiki/Nasdaq-100"
        tables = pd.read_html(requests.get(url, headers=headers).text)
        component_table = tables[0]
        col = 'Symbol' if 'Symbol' in component_table.columns else 'Ticker'
        raw_tickers = [str(s).strip().replace('.', '-') for s in component_table[col].dropna().tolist()]
    except Exception:
        raw_tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "BRK-B", "LLY", "AVGO", "JPM"]
        
    sanitized = sorted(list(set([t.upper() for t in raw_tickers if len(t) <= 5])))
    all_tickers = list(set(sanitized + [etf_target]))
    
    # 3. Pull trailing market data metrics
    print(f"📥 Querying current market pricing across {len(sanitized)} index equities...")
    data = yf.download(all_tickers, period="6mo", interval="1d", auto_adjust=False, progress=False, threads=True)
    close_matrix = data['Close'].ffill().fillna(0.0).round(2)
    valid_assets = [c for c in sanitized if c in close_matrix.columns and c != etf_target]
    
    # 4. Generate momentum metrics
    returns_df = close_matrix[valid_assets].pct_change()
    lookback = 63 
    
    p_start_live = close_matrix[valid_assets].iloc[-lookback]
    p_end_live = close_matrix[valid_assets].iloc[-1]
    returns_live = (p_end_live / p_start_live) - 1
    vols_live = returns_df.iloc[-lookback:].std() * np.sqrt(252)
    live_scores = (returns_live / (vols_live + 1e-8)).dropna()
    live_ranked = [a[0] for a in sorted(live_scores.items(), key=lambda x: (-round(x[1], 4), x[0]))]
    
    # 5. Build clean target trade allocations slip dataframe
    live_rows = []
    for r in range(num_stocks):
        tk = live_ranked[r]
        latest_close_price = float(close_matrix[tk].iloc[-1])
        target_dollar_allocation = portfolio_cash * normalized_weights[r]
        target_shares_count = target_dollar_allocation / latest_close_price if latest_close_price > 0 else 0.0
        
        live_rows.append({
            "Portfolio Position": f"Rank #{r + 1}",
            "Stock Ticker Symbol": tk,
            "Latest Close Price": latest_close_price,  # Stored as numeric float for calculations
            "Target Dollar Amount to Buy": round(target_dollar_allocation, 2),
            "Fractional Shares to Buy": round(target_shares_count, 5)
        })
        
    df_live_signals = pd.DataFrame(live_rows).set_index("Portfolio Position")
    
    # Render clean string representation for console display layout
    df_display = df_live_signals.copy()
    df_display["Latest Close Price"] = df_display["Latest Close Price"].map("${:,.2f}".format)
    df_display["Target Dollar Amount to Buy"] = df_display["Target Dollar Amount to Buy"].map("${:,.2f}".format)
    
    print("\n" + "═"*75)
    print(f"🦅 LIVE ACTIONABLE NEW TARGET ALLOCATIONS FOR {etf_target}")
    print("═"*75)
    display(df_display)
    print("═"*75 + "\n")
    
    # 6. Check environment for prior month reallocations dataset
    df_prior = find_and_parse_prior_month_file()
    
    if df_prior is not None:
        print("\n" + "🔄" + "─"*73)
        print("📊 DYNAMIC PORTFOLIO REBALANCING ROUTINE SLIP (Variance Analysis)")
        print("─"*75)
        
        # Parse data out of historical frames safely converting currency formats back to floats
        prior_positions = {}
        for pos, row in df_prior.iterrows():
            ticker = str(row["Stock Ticker Symbol"])
            dollar_str = str(row["Target Dollar Amount to Buy"]).replace("$", "").replace(",", "")
            shares_str = str(row["Fractional Shares to Buy"])
            prior_positions[ticker] = {
                "Position": pos,
                "Dollars": float(dollar_str),
                "Shares": float(shares_str)
            }
            
        current_positions = {}
        for pos, row in df_live_signals.iterrows():
            ticker = str(row["Stock Ticker Symbol"])
            current_positions[ticker] = {
                "Position": pos,
                "Dollars": float(row["Target Dollar Amount to Buy"]),
                "Shares": float(row["Fractional Shares to Buy"]),
                "Price": float(row["Latest Close Price"])
            }
            
        all_tracked_tickers = set(prior_positions.keys()).union(set(current_positions.keys()))
        rebalance_actions = []
        
        for tk in all_tracked_tickers:
            # Case A: Complete Liquidation (Asset dropped out of momentum entirely)
            if tk in prior_positions and tk not in current_positions:
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": "🔴 FULL LIQUIDATION",
                    "Prior Position": prior_positions[tk]["Position"],
                    "New Position": "Out of Index",
                    "Net Dollar Swap": f"-${prior_positions[tk]['Dollars']:,.2f}",
                    "Net Shares Variance": f"-{prior_positions[tk]['Shares']:.5f}"
                })
            # Case B: Brand New Entry (Asset jumped into momentum top ranks)
            elif tk in current_positions and tk not in prior_positions:
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": "🟢 NEW POSITION ENTRY",
                    "Prior Position": "None",
                    "New Position": current_positions[tk]["Position"],
                    "Net Dollar Swap": f"+${current_positions[tk]['Dollars']:,.2f}",
                    "Net Shares Variance": f"+{current_positions[tk]['Shares']:.5f}"
                })
            # Case C: Dynamic Holding Shift (Rank adjustments / Allocation adjustments)
            else:
                d_diff = current_positions[tk]["Dollars"] - prior_positions[tk]["Dollars"]
                s_diff = current_positions[tk]["Shares"] - prior_positions[tk]["Shares"]
                
                if d_diff == 0:
                    status = "⚪ NO CHANGE"
                    d_str = "$0.00"
                    s_str = "0.00000"
                elif d_diff > 0:
                    status = "⚡ INCREASE POSITION"
                    d_str = f"+${d_diff:,.2f}"
                    s_str = f"+{s_diff:.5f}"
                else:
                    status = "📉 TRIM POSITION"
                    d_str = f"-${abs(d_diff):,.2f}"
                    s_str = f"-{abs(s_diff):.5f}"
                    
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": status,
                    "Prior Position": prior_positions[tk]["Position"],
                    "New Position": current_positions[tk]["Position"],
                    "Net Dollar Swap": d_str,
                    "Net Shares Variance": s_str
                })
                
        df_rebalance_slip = pd.DataFrame(rebalance_actions).set_index("Stock Ticker")
        display(df_rebalance_slip)
        print("─"*75 + "\n")
        
    # 7. Write today's trading log targets cleanly to disk format
    current_date_str = datetime.now().strftime("%m%d%Y")
    export_filename = f"{current_date_str}_Reallocations.csv"
    try:
        df_live_signals.to_csv(export_filename, index=True)
        print(f"💾 [Export System]: Current execution profile logged as: {export_filename}")
    except Exception as e:
        print(f"❌ File Save Failure: {e}")
        
    return df_live_signals

if __name__ == '__main__':
    df_current_orders = execute_production_signals_run(portfolio_cash=USER_PORTFOLIO_VALUE)

🎯 Target Model: ETF=SPY | Buy Density N=10 Stocks | Total Capital=$25,000.00
📡 Resolving underlying constituent matrix for SPY...


C:\Users\reywa\AppData\Local\Temp\ipykernel_15120\3755302829.py:67: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(requests.get(url, headers=headers).text)


📥 Querying current market pricing across 503 index equities...

═══════════════════════════════════════════════════════════════════════════
🦅 LIVE ACTIONABLE NEW TARGET ALLOCATIONS FOR SPY
═══════════════════════════════════════════════════════════════════════════


,Stock Ticker Symbol,Latest Close Price,Target Dollar Amount to Buy,Fractional Shares to Buy
Portfolio Position,,,,
Rank #1,HUM,$376.00,"$6,250.00",16.62234
Rank #2,STX,"$1,025.36","$4,500.00",4.38870
Rank #3,SNDK,"$2,335.00","$3,750.00",1.60600
Rank #4,MU,"$1,213.56","$3,000.00",2.47207
Rank #5,INTC,$132.87,"$2,500.00",18.81538
Rank #6,CNC,$64.77,"$2,000.00",30.87849
Rank #7,AMD,$532.57,"$1,250.00",2.34711
Rank #8,WDC,$675.39,"$1,000.00",1.48063
Rank #9,PANW,$293.09,$500.00,1.70596


═══════════════════════════════════════════════════════════════════════════

📖 [Scanner]: Discovered historical portfolio anchor data sheet: 05232026_Reallocations.csv

🔄─────────────────────────────────────────────────────────────────────────
📊 DYNAMIC PORTFOLIO REBALANCING ROUTINE SLIP (Variance Analysis)
───────────────────────────────────────────────────────────────────────────


,Action Status,Prior Position,New Position,Net Dollar Swap,Net Shares Variance
Stock Ticker,,,,,
PANW,⚪ NO CHANGE,Rank #9,Rank #9,$0.00,0.00000
UNH,🟢 NEW POSITION ENTRY,None,Rank #10,+$250.00,+0.60164
CNC,⚪ NO CHANGE,Rank #6,Rank #6,$0.00,0.00000
STX,⚪ NO CHANGE,Rank #2,Rank #2,$0.00,0.00000
HUM,⚪ NO CHANGE,Rank #1,Rank #1,$0.00,0.00000
INTC,⚡ INCREASE POSITION,Rank #7,Rank #5,"+$1,250.00",+9.40767
WDC,⚪ NO CHANGE,Rank #8,Rank #8,$0.00,0.00000
AMD,📉 TRIM POSITION,Rank #5,Rank #7,"-$1,250.00",-2.34711
MU,⚪ NO CHANGE,Rank #4,Rank #4,$0.00,0.00000


───────────────────────────────────────────────────────────────────────────

💾 [Export System]: Current execution profile logged as: 06272026_Reallocations.csv


In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import json
import os
import glob
from datetime import datetime

# =====================================================================
# 🦅 LIVE PRODUCTION TRADE CONFIGURATIONS (Modify your capital here)
# =====================================================================
USER_PORTFOLIO_VALUE = 25000.00  # Total cash dollar balance to invest
# =====================================================================

def find_and_parse_prior_month_file() -> pd.DataFrame:
    """
    Scans the current workspace directory for any reallocation CSV files
    generated during the previous calendar month.
    """
    today = datetime.now()
    
    # Calculate previous month and year values handling January rollbacks safely
    prev_month = 12 if today.month == 1 else today.month - 1
    prev_year = today.year - 1 if today.month == 1 else today.year
    
    # Format pattern to match matching month string sequence (e.g. "05??2026_Reallocations.csv")
    match_pattern = f"{prev_month:02d}??{prev_year}_Reallocations.csv"
    discovered_files = glob.glob(match_pattern)
    
    if not discovered_files:
        print(f"🔍 [Scanner]: No historical reallocations ledger found matching target pattern '{match_pattern}'.")
        return None
        
    # Sort files to isolate the latest logging file if multiple match the pattern
    target_file = sorted(discovered_files)[-1]
    print(f"📖 [Scanner]: Discovered historical portfolio anchor data sheet: {target_file}")
    
    # Read and ensure index tracking is correctly parsed
    df_prior = pd.read_csv(target_file)
    if "Portfolio Position" in df_prior.columns:
        df_prior = df_prior.set_index("Portfolio Position")
    return df_prior


def execute_production_signals_run(portfolio_cash: float):
    target_csv = "Dynamic_etf_optimized_variables.csv"
    
    if not os.path.exists(target_csv):
        raise FileNotFoundError(f"❌ Aborted: Configuration ledger file '{target_csv}' missing. Run optimizer first.")
        
    df_variables = pd.read_csv(target_csv)
    etf_target  = str(df_variables["etf_target"].iloc[0]).upper().strip()
    num_stocks  = int(df_variables["num_stocks"].iloc[0])
    allocations = json.loads(df_variables["allocations"].iloc[0])
    
    total_w_sum = sum(allocations) if sum(allocations) > 0 else 100
    normalized_weights = [w / total_w_sum for w in allocations]
    
    print(f"🎯 Target Model: ETF={etf_target} | Buy Density N={num_stocks} Stocks | Total Capital=${portfolio_cash:,.2f}")
    
    # 2. Extract full-basket composition from index repositories
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print(f"📡 Resolving underlying constituent matrix for {etf_target}...")
    try:
        url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies" if "SPY" in etf_target else "https://en.wikipedia.org/wiki/Nasdaq-100"
        tables = pd.read_html(requests.get(url, headers=headers).text)
        component_table = tables[0]
        col = 'Symbol' if 'Symbol' in component_table.columns else 'Ticker'
        raw_tickers = [str(s).strip().replace('.', '-') for s in component_table[col].dropna().tolist()]
    except Exception:
        raw_tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "BRK-B", "LLY", "AVGO", "JPM"]
        
    sanitized = sorted(list(set([t.upper() for t in raw_tickers if len(t) <= 5])))
    all_tickers = list(set(sanitized + [etf_target]))
    
    # 3. Pull trailing market data metrics
    print(f"📥 Querying current market pricing across {len(sanitized)} index equities...")
    data = yf.download(all_tickers, period="6mo", interval="1d", auto_adjust=False, progress=False, threads=True)
    close_matrix = data['Close'].ffill().fillna(0.0).round(2)
    valid_assets = [c for c in sanitized if c in close_matrix.columns and c != etf_target]
    
    # 4. Generate momentum metrics
    returns_df = close_matrix[valid_assets].pct_change()
    lookback = 63 
    
    p_start_live = close_matrix[valid_assets].iloc[-lookback]
    p_end_live = close_matrix[valid_assets].iloc[-1]
    returns_live = (p_end_live / p_start_live) - 1
    vols_live = returns_df.iloc[-lookback:].std() * np.sqrt(252)
    live_scores = (returns_live / (vols_live + 1e-8)).dropna()
    live_ranked = [a[0] for a in sorted(live_scores.items(), key=lambda x: (-round(x[1], 4), x[0]))]
    
    # 5. Build clean target trade allocations slip dataframe
    live_rows = []
    for r in range(num_stocks):
        tk = live_ranked[r]
        latest_close_price = float(close_matrix[tk].iloc[-1])
        target_dollar_allocation = portfolio_cash * normalized_weights[r]
        target_shares_count = target_dollar_allocation / latest_close_price if latest_close_price > 0 else 0.0
        
        live_rows.append({
            "Portfolio Position": f"Rank #{r + 1}",
            "Stock Ticker Symbol": tk,
            "Latest Close Price": latest_close_price,  
            "Target Dollar Amount to Buy": round(target_dollar_allocation, 2),
            "Fractional Shares to Buy": round(target_shares_count, 5)
        })
        
    df_live_signals = pd.DataFrame(live_rows).set_index("Portfolio Position")
    
    # Render clean string representation for console display layout
    df_display = df_live_signals.copy()
    df_display["Latest Close Price"] = df_display["Latest Close Price"].map("${:,.2f}".format)
    df_display["Target Dollar Amount to Buy"] = df_display["Target Dollar Amount to Buy"].map("${:,.2f}".format)
    
    print("\n" + "═"*75)
    print(f"🦅 LIVE ACTIONABLE NEW TARGET ALLOCATIONS FOR {etf_target}")
    print("═"*75)
    display(df_display)
    print("═"*75 + "\n")
    
    # 6. Check environment for prior month reallocations dataset
    df_prior = find_and_parse_prior_month_file()
    
    if df_prior is not None:
        print("\n" + "🔄" + "─"*73)
        print("📊 DYNAMIC PORTFOLIO REBALANCING ROUTINE SLIP (Variance Analysis)")
        print("─"*75)
        
        prior_positions = {}
        for pos, row in df_prior.iterrows():
            ticker = str(row["Stock Ticker Symbol"])
            dollar_str = str(row["Target Dollar Amount to Buy"]).replace("$", "").replace(",", "")
            shares_str = str(row["Fractional Shares to Buy"])
            prior_positions[ticker] = {
                "Position": pos,
                "Dollars": float(dollar_str),
                "Shares": float(shares_str)
            }
            
        current_positions = {}
        for pos, row in df_live_signals.iterrows():
            ticker = str(row["Stock Ticker Symbol"])
            current_positions[ticker] = {
                "Position": pos,
                "Dollars": float(row["Target Dollar Amount to Buy"]),
                "Shares": float(row["Fractional Shares to Buy"]),
                "Price": float(row["Latest Close Price"])
            }
            
        all_tracked_tickers = set(prior_positions.keys()).union(set(current_positions.keys()))
        rebalance_actions = []
        
        for tk in all_tracked_tickers:
            if tk in prior_positions and tk not in current_positions:
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": "🔴 FULL LIQUIDATION",
                    "Prior Position": prior_positions[tk]["Position"],
                    "New Position": "Out of Index",
                    "Net Dollar Swap": f"-${prior_positions[tk]['Dollars']:,.2f}",
                    "Net Shares Variance": f"-{prior_positions[tk]['Shares']:.5f}"
                })
            elif tk in current_positions and tk not in prior_positions:
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": "🟢 NEW POSITION ENTRY",
                    "Prior Position": "None",
                    "New Position": current_positions[tk]["Position"],
                    "Net Dollar Swap": f"+${current_positions[tk]['Dollars']:,.2f}",
                    "Net Shares Variance": f"+{current_positions[tk]['Shares']:.5f}"
                })
            else:
                d_diff = current_positions[tk]["Dollars"] - prior_positions[tk]["Dollars"]
                s_diff = current_positions[tk]["Shares"] - prior_positions[tk]["Shares"]
                
                if d_diff == 0:
                    status = "⚪ NO CHANGE"
                    d_str = "$0.00"
                    s_str = "0.00000"
                elif d_diff > 0:
                    status = "⚡ INCREASE POSITION"
                    d_str = f"+${d_diff:,.2f}"
                    s_str = f"+{s_diff:.5f}"
                else:
                    status = "📉 TRIM POSITION"
                    d_str = f"-${abs(d_diff):,.2f}"
                    s_str = f"-{abs(s_diff):.5f}"
                    
                rebalance_actions.append({
                    "Stock Ticker": tk,
                    "Action Status": status,
                    "Prior Position": prior_positions[tk]["Position"],
                    "New Position": current_positions[tk]["Position"],
                    "Net Dollar Swap": d_str,
                    "Net Shares Variance": s_str
                })
                
        df_rebalance_slip = pd.DataFrame(rebalance_actions)
        
        # ⚡️ CHRONOLOGICAL SORTING MATRIX LAYER
        # Extracts digits from "Rank #X". Assigns 999 to "None" entries to force new entries to the bottom.
        df_rebalance_slip['Sort_Key'] = df_rebalance_slip['Prior Position'].apply(
            lambda x: int(''.join(filter(str.isdigit, str(x)))) if any(char.isdigit() for char in str(x)) else 999
        )
        
        # Sort ascending, clean sorting helper keys, and assign Ticker back as the primary index
        df_rebalance_slip = df_rebalance_slip.sort_values(by="Sort_Key").drop(columns=['Sort_Key']).set_index("Stock Ticker")
        
        display(df_rebalance_slip)
        print("─"*75 + "\n")
        
    # 7. Write today's trading log targets cleanly to disk format
    current_date_str = datetime.now().strftime("%m%d%Y")
    export_filename = f"{current_date_str}_Reallocations.csv"
    try:
        df_live_signals.to_csv(export_filename, index=True)
        print(f"💾 [Export System]: Current execution profile logged as: {export_filename}")
    except Exception as e:
        print(f"❌ File Save Failure: {e}")
        
    return df_live_signals

if __name__ == '__main__':
    df_current_orders = execute_production_signals_run(portfolio_cash=USER_PORTFOLIO_VALUE)

🎯 Target Model: ETF=SPY | Buy Density N=10 Stocks | Total Capital=$25,000.00
📡 Resolving underlying constituent matrix for SPY...


C:\Users\reywa\AppData\Local\Temp\ipykernel_15120\3229346886.py:67: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(requests.get(url, headers=headers).text)


📥 Querying current market pricing across 503 index equities...

═══════════════════════════════════════════════════════════════════════════
🦅 LIVE ACTIONABLE NEW TARGET ALLOCATIONS FOR SPY
═══════════════════════════════════════════════════════════════════════════


,Stock Ticker Symbol,Latest Close Price,Target Dollar Amount to Buy,Fractional Shares to Buy
Portfolio Position,,,,
Rank #1,HUM,$376.00,"$6,250.00",16.62234
Rank #2,STX,"$1,025.36","$4,500.00",4.38870
Rank #3,SNDK,"$2,335.00","$3,750.00",1.60600
Rank #4,MU,"$1,213.56","$3,000.00",2.47207
Rank #5,INTC,$132.87,"$2,500.00",18.81538
Rank #6,CNC,$64.77,"$2,000.00",30.87849
Rank #7,AMD,$532.57,"$1,250.00",2.34711
Rank #8,WDC,$675.39,"$1,000.00",1.48063
Rank #9,PANW,$293.09,$500.00,1.70596


═══════════════════════════════════════════════════════════════════════════

📖 [Scanner]: Discovered historical portfolio anchor data sheet: 05232026_Reallocations.csv

🔄─────────────────────────────────────────────────────────────────────────
📊 DYNAMIC PORTFOLIO REBALANCING ROUTINE SLIP (Variance Analysis)
───────────────────────────────────────────────────────────────────────────


,Action Status,Prior Position,New Position,Net Dollar Swap,Net Shares Variance
Stock Ticker,,,,,
HUM,⚪ NO CHANGE,Rank #1,Rank #1,$0.00,0.00000
STX,⚪ NO CHANGE,Rank #2,Rank #2,$0.00,0.00000
SNDK,⚪ NO CHANGE,Rank #3,Rank #3,$0.00,0.00000
MU,⚪ NO CHANGE,Rank #4,Rank #4,$0.00,0.00000
AMD,📉 TRIM POSITION,Rank #5,Rank #7,"-$1,250.00",-2.34711
CNC,⚪ NO CHANGE,Rank #6,Rank #6,$0.00,0.00000
INTC,⚡ INCREASE POSITION,Rank #7,Rank #5,"+$1,250.00",+9.40767
WDC,⚪ NO CHANGE,Rank #8,Rank #8,$0.00,0.00000
PANW,⚪ NO CHANGE,Rank #9,Rank #9,$0.00,0.00000


───────────────────────────────────────────────────────────────────────────

💾 [Export System]: Current execution profile logged as: 06272026_Reallocations.csv
